In [ ]:
import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter
from rdkit import Chem
from rdkit.Chem import inchi

MOLS = ["JNJ_Icotrokinra", "MK-0616", "Daraxonrasib", "cyclosporine", "Simeprevir"]
SHOW = {"JNJ_Icotrokinra": "icotrokinra", "MK-0616": "enlicitide", "Daraxonrasib": "daraxonrasib",
        "cyclosporine": "cyclosporine", "Simeprevir": "simeprevir"}
NAME = {"Icotrokinra": "JNJ_Icotrokinra", "JNJ_Icotrokinra": "JNJ_Icotrokinra",
        "MK-0616": "MK-0616", "Daraxonrasib": "Daraxonrasib", "Cyclosporine": "cyclosporine",
        "cyclosporine": "cyclosporine", "Simeprevir": "Simeprevir"}
ORDER = ["Experiment", "Our model", "ADMET-AI", "ADMETlab 3.0", "DeepPK", "ADMETPred", "admetSAR 3.0"]
MARK = dict(zip(ORDER, ["s", "o", "^", "D", "v", "h", "*"]))
COL = dict(zip(ORDER, ["#000000", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]))
PLOTS = [("Caco2_AB", "Permeability", "log cm/s", (-8, 0), False),
         ("fu", "Plasma unbound fraction", "%", (0, 130), False),
         ("Terminal_t12", "Terminal t½", "h", (.3, 200), True),
         ("F", "Oral bioavailability", "%", (0, 100), False)]
XTICKS = {"fu": [0, 10, 20, 30, 40, 60, 80, 100, 120], "Terminal_t12": [.3, .5, 1, 3, 10, 30, 100, 200]}
WIDE = {"Terminal_t12", "F"}
DY_N = dict(zip(ORDER, [.56, .40, .24, .08, -.08, -.24, -.40]))
LANE_STEP, REP_STEP, BLOCK_N = .78, .06, 3.1
BLOCK_W = (len(ORDER) - 1) * LANE_STEP + 1.7
SRC_I = {s: i for i, s in enumerate(ORDER)}
FU_BINS, T12_BINS, F_BINS = [(0, 1), (1, 10), (10, 20), (20, 50), (50, 100)], [(.3, 3), (3, 6), (6, 12), (12, 200)], [(0, 5), (5, 20), (20, 50), (50, 100)]


def ikey(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None: raise ValueError(f"Invalid SMILES: {str(smiles)[:80]}")
    return inchi.MolToInchiKey(mol)


def molecule_name(value):
    value = str(value).strip()
    if value in NAME: return NAME[value]
    if value in MOLS: return value
    raise KeyError(f"Unknown molecule name: {value!r}")


def rec(mol, ep, src, typ="point", value=np.nan, low=np.nan, high=np.nan, unit="", species="human", route="", reported=""):
    return {"Molecule": mol, "Endpoint": ep, "Source": src, "Species": species, "Route": route,
            "Reported_Output": reported, "Plot_Type": typ, "Plot_Value": value, "Plot_Low": low,
            "Plot_High": high, "Plot_Unit": unit, "Use_in_Plot": "yes"}


def interval(mol, ep, src, low, high, unit, **kw):
    return rec(mol, ep, src, "interval", low=low, high=high, unit=unit, **kw)


def upper(mol, ep, src, high, unit, **kw):
    return rec(mol, ep, src, "upper", high=high, unit=unit, **kw)


def lower(mol, ep, src, low, unit, **kw):
    return rec(mol, ep, src, "lower", low=low, unit=unit, **kw)


def ordinal(mol, ep, src, bins, cls, unit):
    low, high = bins[cls]
    if cls == 0: return upper(mol, ep, src, high, unit)
    if cls == len(bins)-1: return lower(mol, ep, src, low, unit)
    return interval(mol, ep, src, low, high, unit)


def molecule_map(root):
    x = pd.read_csv(root / "data/external_five_case/input.csv")
    return dict(zip(x.InChIKey.astype(str).str.strip(), x["name"].map(molecule_name)))


def attach(df, smiles_col, mapping):
    x = df.copy(); x["Molecule"] = x[smiles_col].map(ikey).map(mapping)
    return x[x.Molecule.notna()]


def experimental(raw):
    x = pd.read_csv(raw / "experimental_ground_truth.csv")
    required = {"Molecule", "Endpoint", "Source", "Plot_Type", "Plot_Value", "Plot_Low", "Plot_High", "Plot_Unit"}
    if missing := required - set(x.columns): raise ValueError(f"experimental_ground_truth.csv missing {sorted(missing)}")
    x["Molecule"] = x.Molecule.map(molecule_name)
    return x[x.Source.astype(str).str.strip().eq("Experiment") & x.Use_in_Plot.astype(str).str.strip().str.lower().eq("yes")].to_dict("records")


def admet_ai(raw, mapping):
    x, rows = attach(pd.read_csv(raw / "ADMET-AI.csv"), "smiles", mapping), []
    for _, r in x.iterrows():
        m = r.Molecule
        rows += [rec(m, "Caco2_AB", "ADMET-AI", value=r.Caco2_Wang, unit="log cm/s"),
                 rec(m, "fu", "ADMET-AI", value=100-r.PPBR_AZ, unit="%"),
                 rec(m, "Terminal_t12", "ADMET-AI", value=r.Half_Life_Obach, unit="h")]
        rows.append(lower(m, "F", "ADMET-AI", 20, "%") if r.Bioavailability_Ma >= .5 else upper(m, "F", "ADMET-AI", 20, "%"))
    return rows


def admetlab(raw, mapping):
    x, rows = attach(pd.read_csv(raw / "ADMETLab3.csv"), "smiles", mapping), []
    for _, r in x.iterrows():
        m = r.Molecule
        rows += [rec(m, "Caco2_AB", "ADMETlab 3.0", value=r.caco2, unit="log cm/s"),
                 rec(m, "fu", "ADMETlab 3.0", value=r.Fu, unit="%"),
                 rec(m, "Terminal_t12", "ADMETlab 3.0", value=r["t0.5"], unit="h")]
        passed = [t for t, col in [(20, "f20"), (30, "f30"), (50, "f50")] if r[col] >= .5]
        rows.append(lower(m, "F", "ADMETlab 3.0", max(passed), "%") if passed else upper(m, "F", "ADMETlab 3.0", 20, "%"))
    return rows


def deeppk(raw, mapping):
    x, rows = attach(pd.read_csv(raw / "deepPK.csv"), "SMILES", mapping), []
    caco, fu = "[Absorption/Caco-2 (logPaap)] Predictions", "[Distribution/Fraction Unbound (Human)] Predictions"
    half, f20, f50 = "[Excretion/Half-Life of Drug] Predictions", "[Absorption/Human Oral Bioavailability 20%] Predictions", "[Absorption/Human Oral Bioavailability 50%] Predictions"
    for _, r in x.iterrows():
        m = r.Molecule
        rows += [rec(m, "Caco2_AB", "DeepPK", value=r[caco], unit="log cm/s"), rec(m, "fu", "DeepPK", value=100*r[fu], unit="%")]
        rows.append(upper(m, "Terminal_t12", "DeepPK", 3, "h") if "< 3" in str(r[half]) else lower(m, "Terminal_t12", "DeepPK", 3, "h"))
        p20, p50 = "Non-" not in str(r[f20]), "Non-" not in str(r[f50])
        if p50: rows.append(lower(m, "F", "DeepPK", 50, "%"))
        elif p20: rows.append(interval(m, "F", "DeepPK", 20, 50, "%"))
        else: rows.append(upper(m, "F", "DeepPK", 20, "%"))
    return rows


def admetpred(raw):
    a, d, rows = pd.read_csv(raw / "admetpred_absorption.csv"), pd.read_csv(raw / "admetpred_distribution.csv"), []
    x = a.merge(d[["Drug_ID", "ppbr"]], on="Drug_ID", validate="one_to_one")
    x["Drug_ID"] = x.Drug_ID.astype(str).str.strip()
    for _, r in x[x.Drug_ID.isin(NAME)].iterrows():
        m = molecule_name(r.Drug_ID)
        rows += [rec(m, "Caco2_AB", "ADMETPred", value=r.caco2, unit="log cm/s"), rec(m, "fu", "ADMETPred", value=100-r.ppbr, unit="%")]
        rows.append(lower(m, "F", "ADMETPred", 20, "%") if int(r.bioavailability) else upper(m, "F", "ADMETPred", 20, "%"))
    return rows


def admetsar(raw, mapping):
    x, rows = attach(pd.read_csv(raw / "admetSAR3.0.txt", sep="\t"), "SMILES", mapping), []
    for _, r in x.iterrows():
        m = r.Molecule
        rows += [rec(m, "Caco2_AB", "admetSAR 3.0", value=r.Caco_2, unit="log cm/s"),
                 rec(m, "fu", "admetSAR 3.0", value=100*(1-r.PPB), unit="%"),
                 rec(m, "Terminal_t12", "admetSAR 3.0", value=10**(-r.T50), unit="h")]
        passed = [t for t, col in [(20, "F20"), (30, "F30"), (50, "F50")] if r[col] >= .5]
        rows.append(lower(m, "F", "admetSAR 3.0", max(passed), "%") if passed else upper(m, "F", "admetSAR 3.0", 20, "%"))
    return rows


def our_model(results):
    rows = []
    x = pd.read_csv(results / "caco2/predictions.csv")
    for _, r in x[x.task.astype(str).str.strip().eq("papp_AtoB")].iterrows(): rows.append(rec(molecule_name(r["name"]), "Caco2_AB", "Our model", value=r.prediction_log_value, unit="log cm/s"))
    x = pd.read_csv(results / "unbound_fraction/predictions.csv")
    for _, r in x[x.task.astype(str).str.strip().eq("human")].iterrows():
        rows.append(ordinal(molecule_name(r["name"]), "fu", "Our model", FU_BINS, int(r.ordinal_class), "%"))
    x = pd.read_csv(results / "terminal_t_half/predictions.csv"); task = "task" if "task" in x else "species_norm"
    for _, r in x[x[task].astype(str).str.strip().eq("human")].iterrows():
        rows.append(ordinal(molecule_name(r["name"]), "Terminal_t12", "Our model", T12_BINS, int(r.ordinal_class), "h"))
    x = pd.read_csv(results / "bioavailability/predictions.csv")
    for _, r in x.iterrows():
        cls = int(r.pred_ge5)+int(r.pred_ge20)+int(r.pred_ge50)
        rows.append(ordinal(molecule_name(r["name"]), "F", "Our model", F_BINS, cls, "%"))
    return rows


def build_data(root, raw, results):
    mapping = molecule_map(root)
    rows = experimental(raw) + our_model(results) + admet_ai(raw, mapping) + admetlab(raw, mapping)
    rows += deeppk(raw, mapping) + admetpred(raw) + admetsar(raw, mapping)
    df = pd.DataFrame(rows)
    for c in ["Plot_Value", "Plot_Low", "Plot_High"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df



ROOT = Path.cwd().resolve()
if ROOT.name == "fig_reproduce": ROOT = ROOT.parent
RAW = ROOT / "fig_reproduce/external_raw"
RESULTS = ROOT / "run_test/external_cases"
df = build_data(ROOT, RAW, RESULTS)
print(f"[DATA] rows={len(df)} | raw={RAW} | results={RESULTS}")
print(df.groupby(["Endpoint", "Source"], observed=True).size().to_string())


In [ ]:
MOLS=["JNJ_Icotrokinra","MK-0616","Daraxonrasib","cyclosporine","Simeprevir"]
SHOW={"JNJ_Icotrokinra":"icotrokinra","cyclosporine":"cyclosporine", "Simeprevir": "simeprevir", "MK-0616": "enlicitide", "Daraxonrasib": "daraxonrasib"}

ORDER=["Experiment","Our model","ADMET-AI","ADMETlab 3.0","DeepPK",
       "ADMETPred","admetSAR 3.0"]

# MARK=dict(zip(ORDER,["s","o","^","D","v","h","*","P","X"]))
MARK=dict(zip(ORDER,["s","o","^","D","v","h","*"]))

COL={
    "Experiment":"#000000",
    "Our model":"#ff7f0e",
    "ADMET-AI":"#2ca02c",
    "ADMETlab 3.0":"#d62728",
    "DeepPK":"#9467bd",
    "ADMETPred":"#8c564b",
    "admetSAR 3.0":"#e377c2",
}

PLOTS=[
    ("Caco2_AB","Permeability","log cm/s",(-8,0),False),
    ("fu","Plasma unbound fraction","%",(0,130),False),
    ("Terminal_t12","Terminal t½","h",(.3,200),True),
    ("F","Oral bioavailability","%",(0,100),False)
]
XTICKS={
    "fu":[0,10,20,30,40,60,80,100,120],
    "Terminal_t12":[.3,.5,1,3,10,30,100,200]
}

WIDE={"Terminal_t12","F"}
DY_N=dict(zip(ORDER,[.56,.40,.24,.08,-.08,-.24,-.40,-.56,-.72]))
LANE_STEP=.78
REP_STEP=.06
BLOCK_N=3.1
BLOCK_W=(len(ORDER)-1)*LANE_STEP+1.7
SRC_I={s:i for i,s in enumerate(ORDER)}

def get_rows(mol,ep,src):
    z=df[(df.Molecule==mol)&(df.Endpoint==ep)&(df.Source==src)]
    return z if ep=="Caco2_AB" or (ep=="F" and src=="Experiment") else z[z.Species=="human"]

def anchor(r):
    if r.Plot_Type=="point": return r.Plot_Value
    if r.Plot_Type=="interval": return (r.Plot_Low+r.Plot_High)/2
    if r.Plot_Type=="upper": return r.Plot_High
    if r.Plot_Type=="lower": return r.Plot_Low
    return np.nan

def ycenters(ep):
    return np.arange(len(MOLS))[::-1]*(BLOCK_W if ep in WIDE else BLOCK_N)

def ypos(ep,yc,src,j=0):
    if ep in WIDE:
        mid=(len(ORDER)-1)/2
        return yc+(mid-SRC_I[src])*LANE_STEP-j*REP_STEP
    return yc+DY_N[src]-j*.03

def draw(ax,r,y,xlim,src):
    typ,lo,hi=r.Plot_Type,*xlim
    c,mk=COL[src],MARK[src]
    size,lw=(52,2.8) if src=="Experiment" else (45,2.2) if src=="Our model" else (30,1.6)

    def point(x,marker=mk,s=size):
        ax.scatter(
            x,y,marker=marker,s=s,zorder=5,
            edgecolors=c,linewidths=1.1,
            facecolors="white" if src=="Our model" else c
        )

    if typ=="point" and pd.notna(r.Plot_Value):
        v=r.Plot_Value
        point(lo,"<",size+7) if v<lo else point(hi,">",size+7) if v>hi else point(v)

    elif typ=="interval" and pd.notna(r.Plot_Low) and pd.notna(r.Plot_High):
        a,b=np.clip(r.Plot_Low,lo,hi),np.clip(r.Plot_High,lo,hi)
        ax.plot([a,b],[y,y],lw=lw,color=c,zorder=2)
        point((a+b)/2)
        if r.Plot_Low<lo: point(lo,"<",size+6)
        if r.Plot_High>hi: point(hi,">",size+6)

    elif typ=="upper" and pd.notna(r.Plot_High):
        b=np.clip(r.Plot_High,lo,hi)
        ax.annotate(
            "",xy=(lo,y),xytext=(b,y),
            arrowprops=dict(arrowstyle="->",lw=lw,color=c,shrinkA=0,shrinkB=0)
        )
        ax.scatter(b,y,marker="|",s=52,color=c,zorder=5)

    elif typ=="lower" and pd.notna(r.Plot_Low):
        a=np.clip(r.Plot_Low,lo,hi)
        ax.annotate(
            "",xy=(hi,y),xytext=(a,y),
            arrowprops=dict(arrowstyle="->",lw=lw,color=c,shrinkA=0,shrinkB=0)
        )
        ax.scatter(a,y,marker="|",s=52,color=c,zorder=5)

    if r.Endpoint=="Terminal_t12" and src=="Experiment" and str(r.Route)=="IV":
        x=anchor(r)
        if pd.notna(x):
            ax.text(
                np.clip(x,lo,hi),y+.6,"IV",
                ha="center",va="bottom",fontsize=7,color="black"
            )

def add_guides(ax,ep,yc):
    if ep in WIDE:
        for y in yc:
            ax.axhline(y,lw=.7,color=".86",zorder=0)
            for s in ORDER:
                ax.axhline(ypos(ep,y,s),lw=.45,color=".94",zorder=0)
    else:
        for y in yc:
            ax.axhline(y,lw=.7,color=".90",zorder=0)

def plot_panel(ax,ep,title,unit,xlim,logx=False):
    yc=ycenters(ep)
    add_guides(ax,ep,yc)

    for i,mol in enumerate(MOLS):
        for src in ORDER:
            for j,(_,r) in enumerate(get_rows(mol,ep,src).iterrows()):
                y=ypos(ep,yc[i],src,j)
                if ep=="Terminal_t12" and src=="Experiment" and str(r.Route)=="IV":
                    y+=.12
                draw(ax,r,y,xlim,src)

    if ep=="Caco2_AB":
        ax.axvline(-6,color=".45",ls="--",lw=1.2,zorder=1)

    if logx:
        ax.set_xscale("log")
        ax.set_xticks(XTICKS[ep])
        ax.xaxis.set_major_formatter(ScalarFormatter())
    elif ep in XTICKS:
        ax.set_xticks(XTICKS[ep])

    pad=(len(ORDER)-1)/2*LANE_STEP+.65 if ep in WIDE else .9
    ax.set(xlim=xlim, ylim=(yc[-1]-pad,yc[0]+pad),
        title=title, xlabel=unit)

    ax.set_yticks(yc)
    ax.set_yticklabels([SHOW.get(m,m) for m in MOLS], fontsize=12)

    ax.tick_params(axis="x", labelsize=10)
    ax.tick_params(axis="y", labelsize=10, length=0)
    ax.xaxis.label.set_size(12)
    ax.title.set_size(12)

    ax.grid(axis="x", color=".90", lw=.8)
    ax.spines[["top","right","left"]].set_visible(False)

fig,axs=plt.subplots(2,2,figsize=(11.6,10.4))
for ax,args in zip(axs.ravel(),PLOTS):
    plot_panel(ax,*args)

# ============================================================
# Legends
# Experiment + our model are visually separated from baselines
# ============================================================
def legend_handle(s):
    return Line2D(
        [0],[0],marker=MARK[s],ls="none",ms=6,
        markeredgecolor=COL[s],
        markerfacecolor="white" if s=="Our model" else COL[s],
        label=s
    )

MAIN=["Experiment","Our model"]
BASE=["ADMET-AI","ADMETlab 3.0","DeepPK","ADMETPred","admetSAR 3.0"]

# Primary comparison
fig.legend(
    handles=[legend_handle(s) for s in MAIN],
    title="Primary comparison",
    title_fontsize=10,
    frameon=False,ncol=2,
    loc="upper center",bbox_to_anchor=(.27,1.008),
    fontsize=9,columnspacing=1.2,handletextpad=.4
)

# External baselines
fig.legend(
    handles=[legend_handle(s) for s in BASE],
    title="External models",
    title_fontsize=10,
    frameon=False,ncol=3,
    loc="upper center",bbox_to_anchor=(.68,1.008),
    fontsize=9,columnspacing=1.2,handletextpad=.4
)

plt.tight_layout(rect=[0,0,1,.95])
plt.show()

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

SHEET = "07_Plot_Ready_4EP"
MOLS = ["JNJ_Icotrokinra", "MK-0616", "Daraxonrasib", "cyclosporine", "Simeprevir"]
MODELS = ["Our model", "ADMET-AI", "ADMETlab 3.0", "DeepPK", "ADMETPred", "admetSAR 3.0"]
ENDPOINTS = ["Caco2_AB", "fu", "Terminal_t12", "F"]
CLASSIFICATION_ENDPOINTS = ["fu", "Terminal_t12", "F"]
CATEGORY_BINS = {
    "fu": [("extremely_low", 0, 1), ("low", 1, 10),
           ("low_intermediate", 10, 20), ("medium", 20, 50), ("high", 50, np.inf)],
    "Terminal_t12": [("short", 0, 3), ("medium", 3, 6),
                     ("intermediate", 6, 12), ("long", 12, np.inf)],
    "F": [("extremely_low", 0, 5), ("low", 5, 20),
          ("medium", 20, 50), ("high", 50, np.inf)],
}


def constraint_from_row(row):
    """Reject malformed constraints instead of silently reducing coverage."""
    typ = str(row["Plot_Type"]).strip().lower()

    def number(column):
        try:
            value = float(row[column])
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{column} must be numeric") from exc
        if not np.isfinite(value):
            raise ValueError(f"{column} must be finite")
        return value

    if typ == "point":
        low = high = number("Plot_Value")
    elif typ == "interval":
        low, high = number("Plot_Low"), number("Plot_High")
        if low > high:
            raise ValueError("Plot_Low exceeds Plot_High")
        if low == high:
            typ = "point"
    elif typ == "upper":
        low, high = -np.inf, number("Plot_High")
    elif typ == "lower":
        low, high = number("Plot_Low"), np.inf
    else:
        raise ValueError(f"Unknown Plot_Type: {typ!r}")
    return {"kind": typ, "value": low if typ == "point" else None,
            "low": low, "high": high}


def contains(c, value):
    if c["kind"] == "upper":
        return value < c["high"]
    if c["kind"] == "lower":
        return value > c["low"]
    return c["low"] <= value <= c["high"]


def overlap_or_compatible(prediction, reference):
    if prediction is None or reference is None:
        return False
    low = max(prediction["low"], reference["low"])
    high = min(prediction["high"], reference["high"])
    if low < high:
        return True
    return low == high and contains(prediction, low) and contains(reference, low)


def range_aware_error(prediction, reference):
    """Distance is zero for overlap, equal points, and touching bounds."""
    if prediction is None or reference is None:
        return np.nan
    return max(0.0, reference["low"] - prediction["high"],
               prediction["low"] - reference["high"])


def constraint_categories(constraint, endpoint):
    if constraint is None or endpoint not in CATEGORY_BINS:
        return set()
    if constraint["kind"] == "point":
        return {label for label, low, high in CATEGORY_BINS[endpoint]
                if low <= constraint["value"] < high}
    return {label for label, low, high in CATEGORY_BINS[endpoint]
            if max(low, constraint["low"]) < min(high, constraint["high"])}


def category_agreement(prediction, reference, endpoint):
    return bool(constraint_categories(prediction, endpoint)
                & constraint_categories(reference, endpoint))


def score_rows(df, molecule, endpoint, source):
    z = df[(df["Molecule"] == molecule) & (df["Endpoint"] == endpoint)
           & (df["Source"] == source)]
    if endpoint == "Caco2_AB" or (endpoint == "F" and source == "Experiment"):
        return z
    return z[z["Species"] == "human"]


def load_data(path=None):
    df = globals()["df"].copy()
    required = {"Molecule", "Endpoint", "Source", "Species", "Plot_Type",
                "Plot_Value", "Plot_Low", "Plot_High", "Use_in_Plot", "Plot_Unit"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    df["Excel_row"] = np.arange(len(df)) + 2
    for column in ["Molecule", "Endpoint", "Source", "Species", "Plot_Type",
                   "Use_in_Plot", "Plot_Unit"]:
        df[column] = df[column].fillna("").astype(str).str.strip()
    for column in ["Species", "Plot_Type", "Use_in_Plot"]:
        df[column] = df[column].str.lower()
    return df[df["Use_in_Plot"].eq("yes")].copy()


def score(path=None):
    df = load_data(path)
    diagnostics, cache = [], {}
    # Parse only rows selected by the stated molecule/model/species rules.
    for molecule in MOLS:
        for endpoint in ENDPOINTS:
            for source in ["Experiment"] + MODELS:
                rows = score_rows(df, molecule, endpoint, source)
                parsed = []
                for _, row in rows.iterrows():
                    context = f"Excel row {row.Excel_row}: {molecule}/{endpoint}/{source}"
                    try:
                        c = constraint_from_row(row)
                    except ValueError as exc:
                        raise ValueError(f"{context}: {exc}") from exc
                    if not row.Plot_Unit:
                        raise ValueError(f"{context}: missing Plot_Unit")
                    if endpoint in ["fu", "F"]:
                        # Do not clip invalid percentage predictions into a valid bin.
                        values = [v for v in [c["low"], c["high"]] if np.isfinite(v)]
                        if any(v < 0 or v > 100 for v in values) or (
                            c["kind"] == "upper" and c["high"] <= 0
                        ) or (c["kind"] == "lower" and c["low"] >= 100):
                            diagnostics.append(f"{context}: invalid percentage constraint; excluded")
                            continue
                    if endpoint in CLASSIFICATION_ENDPOINTS:
                        if not constraint_categories(c, endpoint):
                            raise ValueError(f"{context}: constraint intersects no decision category")
                        values = [v for v in [c["low"], c["high"]] if np.isfinite(v)]
                        if any(v < 0 for v in values):
                            raise ValueError(f"{context}: negative duration")
                    parsed.append((c, row.Plot_Unit, int(row.Excel_row)))
                if len(parsed) > 1:
                    diagnostics.append(f"{molecule}/{endpoint}/{source}: {len(parsed)} rows; best pairing used")
                cache[molecule, endpoint, source] = parsed

    records = []
    for model in MODELS:
        for molecule in MOLS:
            for endpoint in ENDPOINTS:
                refs = cache[molecule, endpoint, "Experiment"]
                preds = cache[molecule, endpoint, model]
                if not refs or not preds:
                    reason = "missing reference" if not refs else "missing prediction"
                    diagnostics.append(f"{molecule}/{endpoint}/{model}: {reason}; not scored")
                    continue
                pairs = []
                for pred, pred_unit, pred_row in preds:
                    for ref, ref_unit, ref_row in refs:
                        if pred_unit != ref_unit:
                            raise ValueError(f"Unit mismatch at Excel rows {pred_row}/{ref_row}: "
                                             f"{pred_unit!r} versus {ref_unit!r}")
                        permeability = endpoint == "Caco2_AB"
                        error = range_aware_error(pred, ref) if permeability else np.nan
                        agreement = (overlap_or_compatible(pred, ref) if permeability
                                     else category_agreement(pred, ref, endpoint))
                        pairs.append((int(agreement), error, pred_row, ref_row))
                # For permeability, minimize distance directly; compatibility breaks ties.
                if endpoint == "Caco2_AB":
                    best = min(pairs, key=lambda p: (p[1], -p[0], p[2], p[3]))
                    if not np.isfinite(best[1]):
                        raise ValueError(f"Non-finite permeability error: {model}/{molecule}")
                else:
                    best = max(pairs, key=lambda p: p[0])
                records.append([model, molecule, endpoint, best[0], best[1], best[2], best[3]])

    detail = pd.DataFrame(records, columns=["Model", "Molecule", "Endpoint", "Agreement",
                         "Permeability_error", "Prediction_row", "Reference_row"])
    # Explicit rows and endpoint loops also handle completely missing endpoints/models.
    summaries = []
    for model in MODELS:
        model_rows = detail[detail.Model.eq(model)]
        errors = model_rows.loc[model_rows.Endpoint.eq("Caco2_AB"), "Permeability_error"].astype(float)
        classified = model_rows[model_rows.Endpoint.isin(CLASSIFICATION_ENDPOINTS)]
        n = len(classified)
        result = {"Model": model, "Permeability_n": len(errors),
                  "Permeability_MAE": errors.mean(), "Permeability_median_AE": errors.median(), "Valid_Agreement_n": n,
                  "Agreements": int(classified.Agreement.sum()),
                  "Agreement_rate": classified.Agreement.mean() if n else np.nan}
        for endpoint in CLASSIFICATION_ENDPOINTS:
            values = classified.loc[classified.Endpoint.eq(endpoint), "Agreement"]
            result[f"sum_{endpoint}"] = int(values.sum())
            result[f"count_{endpoint}"] = len(values)
            result[f"{endpoint}_rate"] = values.mean() if len(values) else np.nan
        summaries.append(result)
    paper_table = pd.DataFrame(summaries).sort_values(
        ["Agreement_rate", "Permeability_median_AE", "Permeability_MAE"],
        ascending=[False, True, True], na_position="last", kind="stable").reset_index(drop=True)
    paper_table.insert(0, "Paper_rank", range(1, len(paper_table) + 1))
    return detail, paper_table, diagnostics


# Review tables use the same selected pairs as the score, not a new pairing.
def format_constraint(c):
    if c["kind"] == "point":
        return f'{c["value"]:.8g}'
    if c["kind"] == "upper":
        return f'<{c["high"]:.8g}'
    if c["kind"] == "lower":
        return f'>{c["low"]:.8g}'
    return f'[{c["low"]:.8g}, {c["high"]:.8g}]'


def category_labels(c, endpoint):
    categories = constraint_categories(c, endpoint)
    return ", ".join(label for label, _, _ in CATEGORY_BINS.get(endpoint, [])
                     if label in categories)


def review_tables(df, detail):
    lookup = df.set_index("Excel_row")
    reviews = []
    for _, result in detail.iterrows():
        pred_row = lookup.loc[result.Prediction_row]
        ref_row = lookup.loc[result.Reference_row]
        pred = constraint_from_row(pred_row)
        ref = constraint_from_row(ref_row)
        endpoint = result.Endpoint
        if endpoint == "Caco2_AB":
            status = "Overlap" if result.Agreement else "Non-overlap"
            if result.Agreement:
                reason = "Numeric constraints overlap; range-aware error is zero."
            elif result.Permeability_error == 0:
                reason = "Constraints touch an excluded strict boundary; distance is zero."
            else:
                direction = "above" if pred["low"] > ref["high"] else "below"
                reason = (f'Prediction is {direction} the experimental constraint; '
                          f'distance = {result.Permeability_error:.8g} {ref_row.Plot_Unit}.')
            pred_categories = ref_categories = "Not used for permeability"
        else:
            pred_categories = category_labels(pred, endpoint)
            ref_categories = category_labels(ref, endpoint)
            status = "Match" if result.Agreement else "Mismatch"
            if result.Agreement:
                shared = constraint_categories(pred, endpoint) & constraint_categories(ref, endpoint)
                ordered = [label for label, _, _ in CATEGORY_BINS[endpoint] if label in shared]
                reason = "Shared decision category: " + ", ".join(ordered) + "."
            else:
                reason = (f'No shared decision category: experiment = {ref_categories}; '
                          f'prediction = {pred_categories}.')
        reviews.append({
            "Model": result.Model, "Molecule": result.Molecule, "Endpoint": endpoint,
            "Status": status, "Experiment": format_constraint(ref),
            "Prediction": format_constraint(pred), "Unit": ref_row.Plot_Unit,
            "Experiment_categories": ref_categories, "Prediction_categories": pred_categories,
            "Reason": reason, "Permeability_error": result.Permeability_error,
            "Reference_row": result.Reference_row, "Prediction_row": result.Prediction_row,
            "Experiment_species": ref_row.Species, "Prediction_species": pred_row.Species,
            "Experiment_reported": ref_row.get("Reported_Output", ""),
            "Prediction_reported": pred_row.get("Reported_Output", ""),
            "Experiment_route": ref_row.get("Route", ""),
            "Experiment_notes": ref_row.get("Notes", ""),
            "Prediction_notes": pred_row.get("Notes", ""),
        })
    columns = ["Model", "Molecule", "Endpoint", "Status", "Experiment", "Prediction", "Unit",
               "Experiment_categories", "Prediction_categories", "Reason", "Permeability_error",
               "Reference_row", "Prediction_row", "Experiment_species", "Prediction_species",
               "Experiment_reported", "Prediction_reported", "Experiment_route",
               "Experiment_notes", "Prediction_notes"]
    review_detail = pd.DataFrame(reviews, columns=columns)
    mismatch_table = review_detail[review_detail.Status.eq("Mismatch")].reset_index(drop=True)
    permeability_table = review_detail[review_detail.Endpoint.eq("Caco2_AB")].reset_index(drop=True)

    # Unscored cases are not mismatches. Show their inputs for manual checking.
    scored = set(zip(detail.Model, detail.Molecule, detail.Endpoint))
    exclusions = []
    def summarize(rows):
        return "; ".join(f'{format_constraint(constraint_from_row(r))} {r.Plot_Unit}'
                         for _, r in rows.iterrows()) or "No selected row"
    for model in MODELS:
        for molecule in MOLS:
            for endpoint in ENDPOINTS:
                if (model, molecule, endpoint) in scored:
                    continue
                refs = score_rows(df, molecule, endpoint, "Experiment")
                preds = score_rows(df, molecule, endpoint, model)
                reasons = []
                if refs.empty:
                    reasons.append("No experimental row after plot/species selection")
                if preds.empty:
                    reasons.append("No prediction row after plot/species selection")
                # Successfully scored inputs fail only the percentage exclusion rule
                # here; malformed constraints or units already raise in score().
                if not refs.empty and not preds.empty:
                    reasons.append("No valid pair: percentage constraint outside 0–100% "
                                   "or strict bound allows no value in 0–100%")
                exclusions.append({"Model": model, "Molecule": molecule, "Endpoint": endpoint,
                                   "Experiment": summarize(refs), "Prediction": summarize(preds),
                                   "Reason": "; ".join(reasons),
                                   "Reference_rows": ", ".join(refs.Excel_row.astype(str)),
                                   "Prediction_rows": ", ".join(preds.Excel_row.astype(str))})
    excluded_table = pd.DataFrame(exclusions, columns=["Model", "Molecule", "Endpoint",
        "Experiment", "Prediction", "Reason", "Reference_rows", "Prediction_rows"])
    return review_detail, mismatch_table, permeability_table, excluded_table


detail, paper_table, diagnostics = score()
review_detail, mismatch_table, permeability_table, excluded_table = review_tables(load_data(), detail)
paper_display = paper_table.round(3)

with pd.option_context("display.max_columns", None, "display.max_rows", None,
                       "display.max_colwidth", None):
    print("Model summary")
    display(paper_display)
    print("Classification mismatches: fu, Terminal_t12, F")
    display(mismatch_table[["Model", "Molecule", "Endpoint", "Experiment", "Prediction", "Unit",
                            "Experiment_categories", "Prediction_categories", "Reason",
                            "Reference_row", "Prediction_row"]])
    print("Excluded or missing comparisons: not counted as mismatches")
    display(excluded_table)